# 12. Revision numbers and Table 1

Run last.

- `results/revision_numbers.csv`: all recorded values (proportions 3 decimals, counts as integers)
- `results/REVISION_RESULTS.md`: values grouped by reviewer comment, with source files
- `results/table1_benchmarks.csv`: Table 1
- Quick-mode values flagged at the top of REVISION_RESULTS.md

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd
import src as la
from src import config as cfg, reporting as rp
cfg.banner()
R = cfg.RESULTS

led = rp.read_ledger()
print(f"{len(led)} numbers recorded")
if len(led):
    led["formatted"] = [rp.fmt_p(v) if (v is not None and abs(v) <= 1.5 and not float(v).is_integer())
                        else (rp.fmt_n(v) if v is not None and float(v).is_integer() else rp.fmt_p(v, 3))
                        for v in led.value]
    led["interval"] = [rp.fmt_ci(l, h) if pd.notna(l) and pd.notna(h) else "" for l, h in zip(led.lo, led.hi)]
    led.to_csv(R / "revision_numbers.csv", index=False)

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000
431 numbers recorded


## 1. Table 1 (unit of analysis, majority rate)

In [3]:
parts = []
if (R / "table1_D1.csv").exists():
    parts.append(pd.read_csv(R / "table1_D1.csv"))
if (R / "D2_population.csv").exists():
    p = pd.read_csv(R / "D2_population.csv")
    for _, r in p[p.population.isin(["WAV files", "WAV, parseable time"])].iterrows():
        parts.append(pd.DataFrame([dict(dataset="D2", task="leak vs no-leak", definition=r.population,
                                        unit="file", n=r.n,
                                        class_counts=f"leak={r.n_leak}; noleak={r.n_noleak}",
                                        majority_rate=r.majority_rate, groups=f"{r.n_loggers} loggers/sites")]))
if (R / "table1_D3.csv").exists():
    parts.append(pd.read_csv(R / "table1_D3.csv"))
if parts:
    t1 = pd.concat(parts, ignore_index=True)
    t1.to_csv(R / "table1_benchmarks.csv", index=False)
    print(t1.drop(columns=[c for c in ["definition"] if c in t1]).to_string(index=False))

dataset                                task                                          unit     n                                              class_counts  majority_rate                                        groups
     D1                       T5_five_class               record (one sensor on one date) 70384 normal=22166; out=19731; in=14932; other=7897; noise=5658       0.314930                       49 sites / 2710 sensors
     D1                  TL_leak_vs_nonleak               record (one sensor on one date) 70384                                 nonleak=35721; leak=34663       0.507516                       49 sites / 2710 sensors
     D1                        TR_out_vs_in               record (one sensor on one date) 34663                                       out=19731; in=14932       0.569224                       49 sites / 2710 sensors
     D1                  TA_alarm_vs_normal               record (one sensor on one date) 70384                                 alarm=48218;

## 2. REVISION_RESULTS.md (by reviewer comment)

In [4]:
GROUPS = [
    ("R1-Major 1 / R2-3: signal-free baseline on a common protocol", ["R1-M1", "R2-3"]),
    ("R1-Major 2 / R2-5 / R2-6: D2 populations, Figure 4, clock vs acoustics", ["R1-M2", "R2-5", "R2-6"]),
    ("R1-Major 3 / R2-8: D3 steps in excess terms; condition definition", ["R1-M3", "R2-8"]),
    ("R1-Major 4 / R2-13: selective risk", ["R1-M4", "R2-13"]),
    ("R1-Major 5/6 / R2-9: factorial", ["R1-M5", "R1-M6", "R2-9"]),
    ("R1-Major 7/8 / R2-11 / R2-16: metrics and uncertainty", ["R1-M7", "R1-M8", "R2-11", "R2-16"]),
    ("R1-Major 9 / R2-15: mitigation setting", ["R1-M9", "R2-15"]),
    ("R1-Major 10/11/12 / R2-14: checksums, column inventory, raw format", ["R1-M10", "R1-M11", "R1-M12", "R2-14"]),
    ("R2-1: identity diagnostics", ["R2-1"]),
    ("R2-2: protocols and deployment settings", ["R2-2"]),
    ("R2-4: lrate", ["R2-4"]),
    ("R2-12: conformal", ["R2-12"]),
    ("R1 minor / other", ["R1-minor", "R1", "R1-M(", "consistency", "self-audit", "l.425-427 support"]),
]
def group_of(comments):
    """Section of the first tag of an entry (exact match; 'R1-minor(x)' -> 'R1-minor')."""
    for tag in [c.strip() for c in str(comments or "").split(",") if c.strip()]:
        base = tag.split("(")[0] if not tag.startswith("R1-M(") else "R1-M("
        for gi, (_, tags) in enumerate(GROUPS):
            if base in tags or any(tag.startswith(t) for t in tags if not t.startswith("R")):
                return gi
    return len(GROUPS) - 1

lines = ["# Revision results (generated by notebook 12)", ""]
if len(led) and led.quick_mode.any():
    lines += ["> **WARNING: some numbers below were produced in QUICK mode. Do not report them.**", ""]
if len(led):
    led["group"] = led.comments.apply(group_of)
for gi, (title, tags) in enumerate(GROUPS):
    q = led[led.group == gi] if len(led) else led
    if not len(q):
        continue
    lines += [f"## {title}", "", "| key | value | interval | note | reviewer tags | source |",
              "|---|---|---|---|---|---|"]
    for _, r in q.iterrows():
        lines.append(f"| `{r.key}` | {r.formatted} | {r.interval} | {r.note or ''} | {r.comments} | {r.source} |")
    lines.append("")
(R / "REVISION_RESULTS.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:60]))
print(f"\n... written to {R / 'REVISION_RESULTS.md'} ({len(led)} numbers)")

# Revision results (generated by notebook 12)

## R1-Major 1 / R2-3: signal-free baseline on a common protocol

| key | value | interval | note | reviewer tags | source |
|---|---|---|---|---|---|
| `D2.avc.timed118.AST.acc` | 0.862 | [0.748, 0.944] | bacc=0.8443; interval: logger bootstrap (fold seed 0) | R2-3,R2-6 | E7_acoustic_vs_clock.csv |
| `D2.avc.timed118.AST + clock.acc` | 0.863 | [0.748, 0.944] | bacc=0.8453; interval: logger bootstrap (fold seed 0) | R2-3,R2-6 | E7_acoustic_vs_clock.csv |
| `D2.avc.timed118.HUBERT_BASE.acc` | 0.809 | [0.683, 0.903] | bacc=0.7823; interval: logger bootstrap (fold seed 0) | R2-3,R2-6 | E7_acoustic_vs_clock.csv |
| `D2.avc.timed118.HUBERT_BASE + clock.acc` | 0.813 | [0.700, 0.920] | bacc=0.7852; interval: logger bootstrap (fold seed 0) | R2-3,R2-6 | E7_acoustic_vs_clock.csv |
| `D2.avc.timed118.Mel2023.acc` | 0.780 | [0.645, 0.876] | bacc=0.7583; interval: logger bootstrap (fold seed 0) | R2-3,R2-6 | E7_acoustic_vs_clock.csv |
| `D2.avc.timed11